# Direct MNA / SSN component benchmark

This notebook records small-grid comparisons between each component's original subcomponent implementation and its optional direct/optimized constructor. Legacy constructors remain the default. EMT runs use 0.5 s; most use Δt = 50 μs (10,000 steps), and the DC π-line uses Δt = 10 μs (50,000 steps). The DP shunt uses 0.04 s at Δt = 10 μs (4,000 steps). Timings are arithmetic means of five runs unless marked provisional.

The trace error is the maximum absolute difference over logged values; values are rounded by the CSV logger. Scaled error is max absolute error divided by the largest legacy trace magnitude (or stated physical bases where needed). The DP transient-stability generator timing is the mean of five runs. This is a component-level microbenchmark, not a prediction of large-grid speedup.

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt

rows = [
    ("EMT Ph3 RxLine", 0.071879, 0.065402, 0.0, 0.0, "5 runs"),
    ("EMT Ph3 NetworkInjection", 0.067671, 0.054320, 0.0, 0.0, "5 runs"),
    ("EMT Ph3 Shunt", 0.059923, 0.059201, 0.0, 0.0, "5 runs"),
    ("EMT Ph3 PiLine", 0.068398, 0.059096, 1.00001e-6, 7.41167e-12, "5 runs"),
    ("EMT Ph3 RXLoad", 0.062459, 0.060071, 0.0, 0.0, "5 runs"),
    ("EMT Ph3 Transformer", 0.066036, 0.061276, 1.29e-4, 1.29e-9, "5 runs"),
    ("EMT Ph3 HalfDecouplingLine", 0.113463, 0.104667, 0.0, 0.0, "5 runs"),
    ("EMT DC PiLine", 0.254789, 0.213369, 1.0e-6, 4.48061e-11, "5 runs"),
    ("EMT Ph1 PiLine", 0.048272, 0.040098, 1.0e-6, 8.03574e-12, "5 runs"),
    ("EMT Ph1 VoltageSourceRamp", 0.035816, 0.036327, 0.0, 0.0, "5 runs"),
    (
        "EMT Ph3 SynchronGeneratorIdeal",
        0.053162,
        0.052868,
        1.00001e-6,
        1.22475e-11,
        "5 runs",
    ),
    (
        "EMT Ph3 SynchronGeneratorTrStab",
        0.071287,
        0.072292,
        0.0,
        0.0,
        "1 run — provisional",
    ),
    (
        "EMT Ph3 GFL (no transformer)",
        1.524383,
        1.376703,
        13.458819,
        0.03204,
        "1 run — provisional",
    ),
    (
        "EMT Ph3 AvVoltageSourceInverterDQ (open-loop)",
        0.943857,
        0.776564,
        1.00001e-6,
        1.00338e-11,
        "5 runs",
    ),
    (
        "EMT Ph3 VSIVoltageControlVCO (open-loop)",
        0.704427,
        0.536352,
        1.00001e-6,
        1.00338e-11,
        "5 runs",
    ),
    (
        "DP Ph1 NetworkInjection",
        0.0210690,
        0.0145451,
        0.0,
        0.0,
        "5-run mean; final validation",
    ),
    (
        "DP Ph1 RXLoad (R + L)",
        0.0219721,
        0.0176075,
        0.0,
        0.0,
        "5-run mean; final validation",
    ),
    (
        "DP Ph1 RXLoadSwitch",
        0.0472971,
        0.0407339,
        3.6e-6,
        9.01388e-9,
        "5-run mean; full-trace relative errors V=9.0e-9, I=0",
    ),
    (
        "DP Ph1 Shunt (G + C)",
        0.0225935,
        0.0178435,
        0.0,
        0.0,
        "5-run mean; final validation",
    ),
    ("DP Ph1 PiLine", 0.0393215, 0.0271580, 0.0, 0.0, "5-run mean; final validation"),
    ("DP Ph1 RxLine", 0.0285247, 0.0186816, 0.0, 0.0, "5-run mean; final validation"),
    (
        "DP Ph1 VoltageSourceRamp",
        0.00456271,
        0.00443265,
        0.0,
        0.0,
        "5-run mean; neutral",
    ),
    (
        "DP Ph1 SynchronGeneratorIdeal",
        0.0191803,
        0.00719372,
        0.0,
        0.0,
        "5-run mean; final validation",
    ),
    (
        "DP Ph1 SynchronGeneratorTrStab",
        0.0251949,
        0.0144056,
        0.009697,
        7.7576e-6,
        "5-run mean; base-scaled trace error",
    ),
    ("DP Ph1 PQLoadCS", 0.0128263, 0.0142239, 0.0, 0.0, "5-run mean; direct is slower"),
    (
        "DP Ph1 SVC (0.04 s, pre-control)",
        0.0313078,
        0.000773717,
        0.0,
        0.0,
        "5-run mean; before control acts",
    ),
    (
        "DP Ph3 NetworkInjection",
        0.0234781,
        0.0190960,
        0.0,
        0.0,
        "5-run mean; final validation",
    ),
    ("DP Ph3 PiLine", 0.260778, 0.101998, 0.0, 0.0, "5-run mean; final validation"),
    (
        "DP Ph1 Transformer (losses enabled)",
        0.0537953,
        0.0393733,
        0.000873139,
        0.000334667,
        "5-run mean; base-scaled errors; noisy legacy",
    ),
    (
        "DP Ph3 Transformer",
        0.275712,
        0.114459,
        0.000502309,
        0.000352833,
        "5-run mean; base-scaled trace errors",
    ),
    (
        "DP Ph1 AvVoltageSourceInverterDQ (no transformer)",
        0.291903,
        0.274011,
        0.0,
        0.0,
        "5-run mean; open-loop and control-on 5 ms traces exact",
    ),
]
df = pd.DataFrame(
    rows,
    columns=[
        "Component",
        "Before fix / legacy mean (s)",
        "After fix / direct mean (s)",
        "Max abs trace error",
        "Scaled trace error",
        "Timing sample",
    ],
)
df["Speedup (%)"] = (
    100
    * (df["Before fix / legacy mean (s)"] - df["After fix / direct mean (s)"])
    / df["Before fix / legacy mean (s)"]
)
display(
    df.style.format(
        {
            "Before fix / legacy mean (s)": "{:.6f}",
            "After fix / direct mean (s)": "{:.6f}",
            "Max abs trace error": "{:.3g}",
            "Scaled trace error": "{:.3g}",
            "Speedup (%)": "{:+.2f}%",
        }
    )
)

In [ ]:
plot_df = df[df["Timing sample"].str.startswith("5")].set_index("Component")
ax = plot_df[["Before fix / legacy mean (s)", "After fix / direct mean (s)"]].plot.bar(
    figsize=(13, 5), color=["#6b7280", "#2563eb"]
)
ax.set_ylabel("Wall time (s)")
ax.set_title("Legacy vs direct component implementation — five-run means")
ax.tick_params(axis="x", rotation=35)
ax.grid(axis="y", alpha=0.25)
plt.tight_layout()

## Interpretation and coverage

Direct mode improves most passive EMT cases in this small setup, while the ramp source and ideal generator are effectively neutral. HalfDecouplingLine's direct path is close to neutral and sensitive to run-to-run noise. Use the per-run average and numerical trace check together; do not infer a grid-scale result from these short runs.

The table labels the legacy implementation as before-fix and the direct implementation as after-fix. DP means are from the latest five-run rerun at 0.04 s and 10 μs. Most direct paths are faster, while DP PQLoadCS is slower in this small fixture. SVC timing covers only the pre-control interval and is not a claim about dynamic-controller performance or equivalence. EMT rows retain their recorded five-run means; the latest broad EMT accuracy/smoke executable was rerun once per pair, but its single-run timings are not substituted into the table. GFL remains provisional, and HalfDecouplingLine is near-neutral/noisy.

## Remaining DP composite coverage

| Component | Direct/no-electrical-child path | Validation status |
|---|---|---|
| DP Ph1 AvVoltageSourceInverterDQ | Implemented for no-transformer case | Open-loop 0.04 s traces and controlled 5 ms traces match; transformer-enabled direct mode is explicitly unsupported and retains the legacy constructor |
| DP Ph1 RXLoadSwitch | Implemented | 0.04 s full trace comparison passes (relative error 9.0e-9 voltage, zero current); five-run mean is 47.5 ms legacy vs 42.0 ms direct |
| DP Ph1 SVC | Present in working tree | Pre-control 0.04 s traces match; beyond 0.1 s the legacy model itself becomes unstable for the tested offset, so dynamic-control parity is not claimed |

Transformer-enabled AvVoltageSourceInverterDQ remains outside timing claims until its direct implementation can be validated.